### Capstone Two: Preprocessing and Training Data Development

In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import joblib


#### Load Clean Dataset (From EDA Phase)

In [2]:
df = pd.read_csv(
    "m5_clean_merged_long.csv",
    nrows=2_000_000   # load first 2 million rows, originnaly has 58 million rows
)

# for complete data I can run- df = pd.read_csv("m5_clean_merged_long.csv", low_memory=False)


df["date"] = pd.to_datetime(df["date"])

df.head()

,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,date,wm_yr_wk,...,event_type_2,snap_CA,snap_TX,snap_WI,sell_price_x,sell_price_y,sell_price,day,day_of_week,week_of_year
0,FOODS_1_001_CA_1_validation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1,3,2011-01-29,11101,...,NaN,0,0,0,2.0,2.0,2.0,29,5,4
1,FOODS_1_001_CA_1_validation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_2,0,2011-01-30,11101,...,NaN,0,0,0,2.0,2.0,2.0,30,6,4
2,FOODS_1_001_CA_1_validation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_3,0,2011-01-31,11101,...,NaN,0,0,0,2.0,2.0,2.0,31,0,5
3,FOODS_1_001_CA_1_validation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_4,1,2011-02-01,11101,...,NaN,1,1,0,2.0,2.0,2.0,1,1,5
4,FOODS_1_001_CA_1_validation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_5,4,2011-02-02,11101,...,NaN,1,0,1,2.0,2.0,2.0,2,2,5


#### Select Modeling Columns

In [3]:
model_df = df[
    [
        "sales",
        "date",
        "sell_price",
        "store_id",
        "cat_id",
        "day_of_week",
        "month",
        "year",
        "week_of_year",
        "snap_CA",
        "snap_TX",
        "snap_WI"
    ]
].copy()

#### Create Dummy Variables

In [4]:
categorical_cols = ["store_id", "cat_id", "day_of_week", "month"]

model_df = pd.get_dummies(
    model_df,
    columns=categorical_cols,
    drop_first=True)

model_df.head()

,sales,date,sell_price,year,week_of_year,snap_CA,snap_TX,snap_WI,day_of_week_1,day_of_week_2,...,month_3,month_4,month_5,month_6,month_7,month_8,month_9,month_10,month_11,month_12
0,3,2011-01-29,2.0,2011,4,0,0,0,False,False,...,False,False,False,False,False,False,False,False,False,False
1,0,2011-01-30,2.0,2011,4,0,0,0,False,False,...,False,False,False,False,False,False,False,False,False,False
2,0,2011-01-31,2.0,2011,5,0,0,0,False,False,...,False,False,False,False,False,False,False,False,False,False
3,1,2011-02-01,2.0,2011,5,1,1,0,True,False,...,False,False,False,False,False,False,False,False,False,False
4,4,2011-02-02,2.0,2011,5,1,0,1,False,True,...,False,False,False,False,False,False,False,False,False,False


#### Time-Based Train/Test Split

In [5]:
model_df = model_df.sort_values("date")

split_index = int(len(model_df) * 0.8)

train_df = model_df.iloc[:split_index]
test_df = model_df.iloc[split_index:]

X_train = train_df.drop(["sales", "date"], axis=1)
y_train = train_df["sales"]

X_test = test_df.drop(["sales", "date"], axis=1)
y_test = test_df["sales"]

#### Standardize Numeric Features

In [6]:
numeric_cols = ["sell_price", "year", "week_of_year"]

scaler = StandardScaler()

X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

#### Confirming Shapes

In [7]:
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (1600000, 23)
X_test shape: (400000, 23)


#### Saving Processed Data

In [8]:
X_train.to_csv("X_train_processed.csv", index=False)
X_test.to_csv("X_test_processed.csv", index=False)
y_train.to_csv("y_train.csv", index=False)
y_test.to_csv("y_test.csv", index=False)

joblib.dump(scaler, "scaler.pkl")

['scaler.pkl']